In [1]:
# ============================================================
# anvl_lcm_windmills_2class_benchmark.ipynb
# ANVL + BGW multi-start benchmark for a two-class latent class model
#
# This notebook estimates a two-class latent class model for the
# data-windmills choice experiment using an XLA-backed ANVL likelihood
# and analytical Jacobian. BGW is used as the external optimiser.
#
# Scope:
# - Configurable CPU or CUDA execution through device_use.
# - Class-specific choice utilities and class-membership probabilities.
# - One fixed-start estimation using Apollo-consistent starting values.
# - Reproducible random-start estimations for the remaining runs.
# - Separate execution of fixed-start and random-start estimations.
# - Per-run checkpointing with failure-tolerant continuation.
# - Successful and failed runs retained in the run-level results.
# - Best-run selection among successfully completed estimations.
# - Separate timing summaries for all, successful, and failed runs.
# - Coefficient export for the best-performing model.
# - Session and hardware information recorded for reproducibility.
#
# Main outputs:
# - results/raw/anvl_lcm_windmills_2class_*_run_results.csv
# - results/raw/anvl_lcm_windmills_2class_*_run_*.rds
# - results/raw/anvl_lcm_windmills_2class_*_best_run.rds
# - results/raw/anvl_lcm_windmills_2class_*_best_run_coef.csv
# - results/benchmark_summary_anvl_*.csv
# - results/session_info_anvl_*.txt
# - results.zip
#
# The benchmark summary reports run counts and estimation time
# separately for all, successful, and failed runs, while per-run timing
# statistics and best-model selection are based on successful runs only.
# ============================================================

In [2]:
# -------------------------------------------------------------------
# 1. Packages
# -------------------------------------------------------------------
# Packages are installed only when missing. The benchmark depends on
# ANVL for XLA-backed tensor evaluation, CUDA/PJRT for GPU execution,
# maxLik and BGW for optimisation, and randtoolbox for Halton draws.

apt_pkgs <- c(
  "build-essential", "cmake", "pkg-config",
  "libprotobuf-dev", "libprotoc-dev", "protobuf-compiler",
  "libuv1-dev", "libcurl4-openssl-dev", "libssl-dev", "libxml2-dev"
)

stopifnot(system2("apt-get", c("update", "-qq")) == 0L)
stopifnot(system2("apt-get", c("install", "-y", apt_pkgs)) == 0L)

# 2. R packages

Sys.setenv(
  CUDA_COMPONENTS = "nvshmem,nccl,nvcc",
  PJRT_INSTALL    = "1"
)

options(repos = c(
  "https://r-xla.r-universe.dev",
  "https://mlverse.r-universe.dev",
  "https://cloud.r-project.org"
))

pkgs <- c(
  "randtoolbox", "bgw", "maxLik",
  "cuda12.8", "pjrt", "anvl"
)

for (pkg in pkgs) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg)
  }
}

# 3. Load packages

suppressPackageStartupMessages({
  library(randtoolbox)
  library(bgw)
  library(maxLik)
  library(pjrt)
  library(anvl)
})


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘rngWELL’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘zoo’, ‘miscTools’, ‘sandwich’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘checkmate’, ‘safetensors’, ‘tengen’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘stablehlo’, ‘xlamisc’




In [3]:
# -------------------------------------------------------------------
# 2. Project-root resolution
# -------------------------------------------------------------------
# Run this notebook from the project root.

proj_root <- getwd()
raw_results_dir <- file.path(proj_root, "results", "raw")

dir.create(raw_results_dir, recursive = TRUE, showWarnings = FALSE)



In [4]:
# -------------------------------------------------------------------
# 3. Define benchmark settings
# -------------------------------------------------------------------
# Change only this line to switch between CPU and CUDA execution.


device_use <- "cuda"

seed <- 1234L
n_reps <- 500L


max_iterations <- 5000L
max_function_evals <- 5000L

if (identical(device_use, "cuda")) {
  install.packages(setdiff("cuda12.8", rownames(installed.packages())))
}

backend_name <- toupper(device_use)
model_name <- paste0("anvl_lcm_windmills_2class_", device_use)

summary_file <- file.path(
  proj_root,
  "results",
  paste0("benchmark_summary_anvl_", device_use, ".csv")
)

session_info_file <- file.path(
  proj_root,
  "results",
  paste0("session_info_anvl_", device_use, ".txt")
)



In [5]:
# -------------------------------------------------------------------
# 4. Load data and print benchmark information
# -------------------------------------------------------------------
# The data are kept in the same wide format used by the Apollo LCM code.

data_url <- "https://raw.githubusercontent.com/edsandorf/evdce/main/Data/data-windmills.csv"

database <- read.csv(data_url, header = TRUE)
database <- database[!is.na(database$choice), ]
database <- database[order(database$id_individual), ]
row.names(database) <- NULL

for (k in 1:3) {
  database[[paste0("choice", k)]] <- as.numeric(database$choice == k)
}

database$educ_2 <- as.numeric(database$education == 2)
database$educ_3 <- as.numeric(database$education == 3)

ids <- unique(database$id_individual)
indiv_index <- match(database$id_individual, ids)
task_index <- as.integer(ave(database$choice, database$id_individual, FUN = seq_along))
panel <- database[match(ids, database$id_individual), ]

n_indiv <- length(ids)
n_task <- max(task_index)
n_obs <- nrow(database)

cat("Benchmark settings\n")
cat("Seed         :", seed, "\n")
cat("Reps         :", n_reps, "\n")
cat("Device       :", device_use, "\n")
cat("Individuals  :", n_indiv, "\n")
cat("Tasks/person :", n_task, "\n")
cat("Observations :", n_obs, "\n")
cat("Raw output   :", raw_results_dir, "\n")
cat("Summary file :", summary_file, "\n")

cat("\nANVL device check:\n")
print(nv_array(1, dtype = "f64", device = device_use))

cpu_info <- tryCatch(
  system("lscpu", intern = TRUE),
  error = function(e) paste("lscpu failed:", conditionMessage(e))
)

cat("\nCPU information:\n")
cat(paste(cpu_info, collapse = "\n"), "\n")

if (identical(device_use, "cuda")) {
  gpu_info <- tryCatch(
    system("nvidia-smi", intern = TRUE),
    error = function(e) paste("nvidia-smi failed:", conditionMessage(e))
  )
} else {
  gpu_info <- character(0)
}

sink(session_info_file)
cat("R session information\n")
print(sessionInfo())

cat("\nPackage versions\n")
for (pkg in c("bgw", "pjrt", "anvl", "cuda12.8")) {
  if (pkg %in% rownames(installed.packages())) {
    cat(pkg, ": ", as.character(packageVersion(pkg)), "\n", sep = "")
  }
}

cat("\nANVL device check\n")
print(nv_array(1, dtype = "f64", device = device_use))

cat("\nCPU information\n")
cat(paste(cpu_info, collapse = "\n"), "\n")

if (length(gpu_info) > 0L) {
  cat("\nGPU information\n")
  cat(paste(gpu_info, collapse = "\n"), "\n")
}
sink()

cat("Session and hardware information saved to:", session_info_file, "\n")



Benchmark settings
Seed         : 1234 
Reps         : 500 
Device       : cuda 
Individuals  : 1000 
Tasks/person : 10 
Observations : 8633 
Raw output   : /content/results/raw 
Summary file : /content/results/benchmark_summary_anvl_cuda.csv 

ANVL device check:


<https://github.com/zml/pjrt-artifacts/releases/download/v14.0.1/pjrt-cuda_linux-amd64.tar.gz>


AnvlArray
 1
[ CUDAf64{1} ] 

CPU information:
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
CPU family:                              6
Model:                                   85
Thread(s) per core:                      2
Core(s) per socket:                      1
Socket(s):                               1
Stepping:                                3
BogoMIPS:                                4000.33
Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good n

In [6]:
# -------------------------------------------------------------------
# 5. Convert database to ANVL arrays
# -------------------------------------------------------------------

make_database_nv <- function(database, device = device_use) {

  ids <- unique(database$id_individual)
  indiv <- match(database$id_individual, ids)
  task <- as.integer(ave(seq_along(database$id_individual),
                         database$id_individual,
                         FUN = seq_along))

  panel <- database[match(ids, database$id_individual), ]

  n_indiv <- length(ids)
  n_task <- max(task)

  make_x <- function(x) {
    m <- matrix(0, n_task, n_indiv)
    m[cbind(task, indiv)] <- as.numeric(x)

    nv_array(m, dtype = "f64", device = device)
  }

  make_z <- function(x) {
    nv_array(as.numeric(x), dtype = "f64", device = device)
  }

  list(
    choice1 = make_x(database$choice == 1),
    choice2 = make_x(database$choice == 2),
    choice3 = make_x(database$choice == 3),

    alt1_sq = make_x(database$alt1_sq),

    alt2_farm2    = make_x(database$alt2_farm2),
    alt2_farm3    = make_x(database$alt2_farm3),
    alt2_height2  = make_x(database$alt2_height2),
    alt2_height3  = make_x(database$alt2_height3),
    alt2_redkite  = make_x(database$alt2_redkite),
    alt2_distance = make_x(database$alt2_distance),
    alt2_cost     = make_x(database$alt2_cost),

    alt3_farm2    = make_x(database$alt3_farm2),
    alt3_farm3    = make_x(database$alt3_farm3),
    alt3_height2  = make_x(database$alt3_height2),
    alt3_height3  = make_x(database$alt3_height3),
    alt3_redkite  = make_x(database$alt3_redkite),
    alt3_distance = make_x(database$alt3_distance),
    alt3_cost     = make_x(database$alt3_cost),

    age    = make_z(panel$age),
    female = make_z(panel$female),
    educ_2 = make_z(panel$education == 2),
    educ_3 = make_z(panel$education == 3)
  )
}


In [7]:
# -------------------------------------------------------------------
# 6. Starting values
# -------------------------------------------------------------------
# The parameter order matches logitf_anvl_lcm_bgw().

start_value <- c(
  asc_sq_a =  0.10, asc_sq_b =  0.00,

  b_medium_farms_a =  0.10, b_medium_farms_b =  0.00,
  b_small_farms_a  =  0.10, b_small_farms_b  =  0.00,

  b_medium_height_a =  0.10, b_medium_height_b =  0.00,
  b_low_height_a    =  0.10, b_low_height_b    =  0.00,

  b_redkite_a  = -0.10, b_redkite_b  = -0.10,
  b_distance_a =  0.10, b_distance_b =  0.10,
  b_cost_a     = -0.10, b_cost_b     = -0.10,

  delta_a        = 0.10,
  gamma_age_a    = 0.00,
  gamma_female_a = 0.00,
  gamma_educ2_a  = 0.00,
  gamma_educ3_a  = 0.00
)

make_random_start <- function(i, base_seed = seed) {

  seed_i <- base_seed + i - 1L
  set.seed(seed_i)

  theta0 <- c(
    asc_sq_a = runif(1, -1, 1),
    asc_sq_b = runif(1, -1, 1),

    b_medium_farms_a = runif(1, -1, 1),
    b_medium_farms_b = runif(1, -1, 1),
    b_small_farms_a  = runif(1, -1, 1),
    b_small_farms_b  = runif(1, -1, 1),

    b_medium_height_a = runif(1, -1, 1),
    b_medium_height_b = runif(1, -1, 1),
    b_low_height_a    = runif(1, -1, 1),
    b_low_height_b    = runif(1, -1, 1),

    b_redkite_a  = runif(1, -1, 1),
    b_redkite_b  = runif(1, -1, 1),
    b_distance_a = runif(1, -1, 1),
    b_distance_b = runif(1, -1, 1),
    b_cost_a     = runif(1, -1, 1),
    b_cost_b     = runif(1, -1, 1),

    delta_a        = runif(1, -1, 1),
    gamma_age_a    = runif(1, -1, 1),
    gamma_female_a = runif(1, -1, 1),
    gamma_educ2_a  = runif(1, -1, 1),
    gamma_educ3_a  = runif(1, -1, 1)
  )

  theta0
}


In [ ]:
# -------------------------------------------------------------------
# 7. Define and compile ANVL latent class likelihood and Jacobian
# -------------------------------------------------------------------

# This part of the code has been omitted temporarily and will be released upon publication of the corresponding paper.

In [9]:
# -------------------------------------------------------------------
# 8. BGW output helpers
# -------------------------------------------------------------------
# These helpers keep the run cells short and the output format fixed.

coef_table <- function(result) {
  data.frame(
    parameter = names(result$betaStop),
    estimate = as.numeric(result$betaStop),
    row.names = NULL
  )
}

first_value <- function(x, default) {
  c(as.vector(x), default)[1]
}

get_bgw_iterations <- function(result) {
  as.integer(first_value(
    unlist(
      result[c(
        "iterations",
        "nIterations",
        "nIter",
        "iter"
      )],
      use.names = FALSE
    ),
    NA_integer_
  ))
}

get_convergence_detail <- function(result) {
  as.character(first_value(
    unlist(
      result[c(
        "convergence",
        "diagnosis",
        "message",
        "code"
      )],
      use.names = FALSE
    ),
    NA_character_
  ))
}

bgw_control <- function(run_id) {
  list(
    maxIterations = max_iterations,
    maxFunctionEvals = max_function_evals,
    silent = TRUE,
    printLevel = 0,
    writeIter = FALSE,
    modelName = paste0(
      model_name,
      "_run_",
      run_id
    )
  )
}

run_bgw <- function(theta0, run_id, seed_value, start_type) {

  n_loglik_eval <- 0L
  n_jacobian_eval <- 0L

  calcR_counted <- function(theta) {
    n_loglik_eval <<- n_loglik_eval + 1L
    calcR_bgw(theta)
  }

  calcJ_counted <- function(theta) {
    n_jacobian_eval <<- n_jacobian_eval + 1L
    calcJ_bgw(theta)
  }

  # Evaluate and compile the probability function before timing.
  initial_probability <- calcR_bgw(theta0)

  stopifnot(
    length(initial_probability) == n_indiv,
    all(is.finite(initial_probability)),
    all(initial_probability > 0)
  )

  initial_loglik <- sum(log(initial_probability))

  # Evaluate and compile the Jacobian function before timing.
  initial_score <- calcJ_bgw(theta0)

  stopifnot(
    length(dim(initial_score)) == 2L,
    all(
      dim(initial_score) ==
        c(n_indiv, length(theta0))
    ),
    all(is.finite(initial_score))
  )

  est_start <- proc.time()

  result <- bgw::bgw_mle(
    calcR = calcR_counted,
    betaStart = theta0,
    calcJ = calcJ_counted,
    bgw_settings = bgw_control(run_id)
  )

  estimation_time <- as.numeric(
    (proc.time() - est_start)["elapsed"]
  )

  stopifnot(
    isTRUE(result$hasAnalyticGrad)
  )

  result_file <- file.path(
    raw_results_dir,
    paste0(
      model_name,
      "_run_",
      run_id,
      ".rds"
    )
  )

  saveRDS(
    result,
    result_file
  )

  list(
    result = result,
    summary = data.frame(
      package = "ANVL",
      implementation = "anvl_lcm_windmills_2class",
      backend = backend_name,
      optimiser = "BGW",
      seed = seed_value,
      run = run_id,
      start_type = start_type,
      initial_loglik = initial_loglik,
      final_loglik = result$finalLL,
      estimation_time_sec = estimation_time,
      bgw_iterations = get_bgw_iterations(result),
      convergence_detail = get_convergence_detail(result),
      n_loglik_eval = n_loglik_eval,
      n_jacobian_eval = n_jacobian_eval,
      has_analytic_gradient = isTRUE(
        result$hasAnalyticGrad
      ),
      result_file = result_file,
      stringsAsFactors = FALSE
    )
  )
}

In [10]:
# -------------------------------------------------------------------
# 9. Fixed-start BGW estimation
# -------------------------------------------------------------------
# This run uses the Apollo-consistent starting values and the compiled
# ANVL probability Jacobian supplied through calcJ.

fixed_run <- run_bgw(
  theta0 = start_value,
  run_id = 0L,
  seed_value = NA,
  start_type = "apollo_start"
)

fixed_model <- fixed_run$result
fixed_summary <- fixed_run$summary

stopifnot(
  isTRUE(fixed_model$hasAnalyticGrad),
  isTRUE(fixed_summary$has_analytic_gradient),
  fixed_summary$n_jacobian_eval > 0L
)

fixed_estimate <- data.frame(
  parameter = names(start_value),
  initial_value = as.numeric(start_value),
  estimate = as.numeric(fixed_model$betaStop),
  row.names = NULL
)

print(fixed_summary)
print(fixed_estimate)

write.csv(
  fixed_summary,
  file.path(
    raw_results_dir,
    "bgw_fixed_start_summary.csv"
  ),
  row.names = FALSE
)

write.csv(
  fixed_estimate,
  file.path(
    raw_results_dir,
    "bgw_fixed_start_estimates.csv"
  ),
  row.names = FALSE
)

saveRDS(
  fixed_model,
  file.path(
    raw_results_dir,
    "bgw_fixed_start_model.rds"
  )
)

  package            implementation backend optimiser seed run   start_type
1    ANVL anvl_lcm_windmills_2class    CUDA       BGW   NA   0 apollo_start
  initial_loglik final_loglik estimation_time_sec bgw_iterations
1       -8112.02    -6618.793                0.23             29
                                     convergence_detail n_loglik_eval
1 ***** Relative function convergence (favorable) *****            73
  n_jacobian_eval has_analytic_gradient
1              32                  TRUE
                                                    result_file
1 /content/results/raw/anvl_lcm_windmills_2class_cuda_run_0.rds
           parameter initial_value    estimate
1           asc_sq_a           0.1 -0.59168994
2           asc_sq_b           0.0 -1.22611048
3   b_medium_farms_a           0.1  0.23915946
4   b_medium_farms_b           0.0 -0.53439982
5    b_small_farms_a           0.1  0.31541730
6    b_small_farms_b           0.0  0.12841448
7  b_medium_height_a           0.1  0.442

In [11]:
# -------------------------------------------------------------------
# 10. Random-start BGW estimations
# -------------------------------------------------------------------
# Runs 1:n_reps use reproducible random starts. Each result is
# checkpointed immediately, and a failed run does not stop later runs.

run_results <- fixed_summary[0, ]

run_results_file <- file.path(
  raw_results_dir,
  paste0(model_name, "_run_results.csv")
)

if (file.exists(run_results_file)) {
  run_results <- read.csv(
    run_results_file,
    stringsAsFactors = FALSE
  )

  missing_columns <- setdiff(
    names(fixed_summary),
    names(run_results)
  )

  for (column in missing_columns) {
    run_results[[column]] <- NA
  }

  run_results <- run_results[
    , names(fixed_summary),
    drop = FALSE
  ]

  run_results <- run_results[
    run_results$start_type == "random_start" &
      run_results$run %in% seq_len(n_reps),
    ,
    drop = FALSE
  ]
}

run_results <- run_results[
  !duplicated(run_results$run, fromLast = TRUE),
  ,
  drop = FALSE
]
run_results <- run_results[order(run_results$run), , drop = FALSE]

write.csv(run_results, run_results_file, row.names = FALSE)

random_run_ids <- setdiff(
  seq_len(n_reps),
  run_results$run
)

for (run_id in random_run_ids) {

  seed_i <- seed + run_id - 1L
  theta0 <- make_random_start(run_id, seed)
  run_start <- proc.time()

  run_i <- tryCatch(
    run_bgw(
      theta0 = theta0,
      run_id = run_id,
      seed_value = seed_i,
      start_type = "random_start"
    ),
    error = function(e) {

      failure_time <- as.numeric(
        (proc.time() - run_start)["elapsed"]
      )

      initial_loglik <- tryCatch(
        sum(log(calcR_bgw(theta0))),
        error = function(e2) NA_real_
      )

      list(
        result = NULL,
        summary = data.frame(
          package = "ANVL",
          implementation = "anvl_lcm_windmills_2class",
          backend = backend_name,
          optimiser = "BGW",
          seed = seed_i,
          run = run_id,
          start_type = "random_start",
          initial_loglik = initial_loglik,
          final_loglik = NA_real_,
          estimation_time_sec = failure_time,
          bgw_iterations = NA_integer_,
          convergence_detail = paste(
            "ERROR:",
            conditionMessage(e)
          ),
          n_loglik_eval = NA_integer_,
          n_jacobian_eval = NA_integer_,
          has_analytic_gradient = NA,
          result_file = NA_character_,
          stringsAsFactors = FALSE
        )
      )
    }
  )

  run_results <- rbind(run_results, run_i$summary)
  run_results <- run_results[order(run_results$run), , drop = FALSE]

  write.csv(run_results, run_results_file, row.names = FALSE)

  run_status <- if (is.finite(run_i$summary$final_loglik)) {
    "completed"
  } else {
    "failed"
  }

  cat("Run", run_id, "of", n_reps, "-", run_status, "\n")
}

random_summary <- run_results[
  run_results$start_type == "random_start",
  ,
  drop = FALSE
]

print(random_summary)

Run 1 of 500 - completed 
Run 2 of 500 - completed 
Run 3 of 500 - completed 
Run 4 of 500 - completed 
Run 5 of 500 - completed 
Run 6 of 500 - completed 
Run 7 of 500 - completed 
Run 8 of 500 - completed 
Run 9 of 500 - completed 
Run 10 of 500 - completed 
Run 11 of 500 - completed 
Run 12 of 500 - completed 
Run 13 of 500 - failed 
Run 14 of 500 - failed 
Run 15 of 500 - completed 
Run 16 of 500 - failed 
Run 17 of 500 - failed 
Run 18 of 500 - failed 
Run 19 of 500 - failed 
Run 20 of 500 - completed 
Run 21 of 500 - failed 
Run 22 of 500 - completed 
Run 23 of 500 - completed 
Run 24 of 500 - failed 
Run 25 of 500 - failed 
Run 26 of 500 - completed 
Run 27 of 500 - completed 
Run 28 of 500 - completed 
Run 29 of 500 - failed 
Run 30 of 500 - completed 
Run 31 of 500 - completed 
Run 32 of 500 - completed 
Run 33 of 500 - completed 
Run 34 of 500 - completed 
Run 35 of 500 - completed 
Run 36 of 500 - completed 
Run 37 of 500 - failed 
Run 38 of 500 - completed 
Run 39 of 500 - 

In [12]:
# -------------------------------------------------------------------
# 11. Summarise outputs
# -------------------------------------------------------------------
# Failed runs remain in the run-level file and are excluded from
# best-model selection and per-run timing statistics, but their
# estimation time is retained in the total-time accounting.

run_results <- read.csv(
  run_results_file,
  stringsAsFactors = FALSE
)
run_results <- run_results[order(run_results$run), , drop = FALSE]

write.csv(
  run_results,
  run_results_file,
  row.names = FALSE
)

model_available <- !is.na(run_results$result_file) &
  nzchar(run_results$result_file) &
  file.exists(run_results$result_file)

is_successful <- is.finite(run_results$final_loglik) & model_available

successful_runs <- run_results[is_successful, , drop = FALSE]
failed_runs <- run_results[!is_successful, , drop = FALSE]

if (nrow(successful_runs) == 0L) {
  stop("No successful BGW run is available for summarisation.")
}

# Runs whose time was never recorded are reported separately so that
# the totals below are not silently understated.
n_missing_time <- sum(is.na(run_results$estimation_time_sec))

if (n_missing_time > 0L) {
  cat(
    "Warning:", n_missing_time,
    "run(s) have no recorded estimation time and are excluded from the totals.\n"
  )
}

best_index <- which.max(successful_runs$final_loglik)
best_run <- successful_runs[best_index, , drop = FALSE]
best_model <- readRDS(best_run$result_file[[1L]])

saveRDS(
  best_model,
  file.path(raw_results_dir, paste0(model_name, "_best_run.rds"))
)

write.csv(
  coef_table(best_model),
  file.path(raw_results_dir, paste0(model_name, "_best_run_coef.csv")),
  row.names = FALSE
)

benchmark_summary <- data.frame(
  package = best_run$package,
  implementation = best_run$implementation,
  backend = best_run$backend,
  optimiser = best_run$optimiser,
  seed = best_run$seed,
  run = best_run$run,
  initial_loglik = best_run$initial_loglik,
  n_runs = nrow(run_results),
  n_successful_runs = nrow(successful_runs),
  n_failed_runs = nrow(failed_runs),

  total_estimation_time_sec = sum(
    run_results$estimation_time_sec,
    na.rm = TRUE
  ),
  total_estimation_time_successful_sec = sum(
    successful_runs$estimation_time_sec,
    na.rm = TRUE
  ),
  total_estimation_time_failed_sec = sum(
    failed_runs$estimation_time_sec,
    na.rm = TRUE
  ),

  mean_estimation_time_sec = mean(successful_runs$estimation_time_sec),
  sd_estimation_time_sec = sd(successful_runs$estimation_time_sec),
  min_estimation_time_sec = min(successful_runs$estimation_time_sec),
  max_estimation_time_sec = max(successful_runs$estimation_time_sec),
  final_loglik = best_run$final_loglik,
  bgw_iterations = best_run$bgw_iterations,
  convergence_detail = best_run$convergence_detail,
  stringsAsFactors = FALSE
)

write.csv(
  benchmark_summary,
  summary_file,
  row.names = FALSE
)

print(benchmark_summary)

cat("\nTotal estimation time over all runs:\n")

cat(
  sprintf(
    "  all runs        : %8.2f sec (%6.2f min, n = %d)\n",
    benchmark_summary$total_estimation_time_sec,
    benchmark_summary$total_estimation_time_sec / 60,
    benchmark_summary$n_runs
  )
)

cat(
  sprintf(
    "  successful runs : %8.2f sec (%6.2f min, n = %d)\n",
    benchmark_summary$total_estimation_time_successful_sec,
    benchmark_summary$total_estimation_time_successful_sec / 60,
    benchmark_summary$n_successful_runs
  )
)

cat(
  sprintf(
    "  failed runs     : %8.2f sec (%6.2f min, n = %d)\n",
    benchmark_summary$total_estimation_time_failed_sec,
    benchmark_summary$total_estimation_time_failed_sec / 60,
    benchmark_summary$n_failed_runs
  )
)

cat("\nSaved run-level results to:\n")
cat(run_results_file, "\n")

cat("\nSaved benchmark summary to:\n")
cat(summary_file, "\n")

cat("\nSaved session information to:\n")
cat(session_info_file, "\n")

zip_file <- file.path(proj_root, "results.zip")

utils::zip(
  zipfile = zip_file,
  files = file.path(proj_root, "results")
)

cat("\nSaved zipped results to:\n")
cat(zip_file, "\n")

  package            implementation backend optimiser seed run initial_loglik
1    ANVL anvl_lcm_windmills_2class    CUDA       BGW 1632 399      -17913.63
  n_runs n_successful_runs n_failed_runs total_estimation_time_sec
1    500               393           107                   118.721
  total_estimation_time_successful_sec total_estimation_time_failed_sec
1                              103.321                             15.4
  mean_estimation_time_sec sd_estimation_time_sec min_estimation_time_sec
1                0.2629033              0.5485613                   0.024
  max_estimation_time_sec final_loglik bgw_iterations
1                   8.384    -6618.793             24
                                     convergence_detail
1 ***** Relative function convergence (favorable) *****

Total estimation time over all runs:
  all runs        :   118.72 sec (  1.98 min, n = 500)
  successful runs :   103.32 sec (  1.72 min, n = 393)
  failed runs     :    15.40 sec (  0.26 min, n = 